# Lecture 4: From Repeated Simulation to Simulated Method of Moments

Everything run in class today lives in this notebook (Python) and its R
counterpart (`04-lecture-examples.Rmd`). No separate lab this time.

# Section 1: Wrapping up the queueing lab

Same simulator as lecture 3, one customer at a time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(681)  # set once, here -- never inside a simulator function


In [ ]:
def simulate_queue(n_customers, lam, mu, rng):
    interarrival_duration = np.empty(n_customers)
    service_duration = np.empty(n_customers)
    arrival_time = np.empty(n_customers)
    service_start_time = np.empty(n_customers)
    departure_time = np.empty(n_customers)

    for i in range(n_customers):
        interarrival_duration[i] = rng.exponential(1.0 / lam)
        if i == 0:
            arrival_time[i] = interarrival_duration[i]
            service_start_time[i] = arrival_time[i]
        else:
            arrival_time[i] = arrival_time[i - 1] + interarrival_duration[i]
            service_start_time[i] = max(arrival_time[i], departure_time[i - 1])
        service_duration[i] = rng.exponential(1.0 / mu)
        departure_time[i] = service_start_time[i] + service_duration[i]

    return service_start_time - arrival_time  # waiting times


Hold `mu` fixed and let `lambda` creep up toward it -- the system gets more utilized (higher traffic intensity $\rho = \lambda/\mu$):

In [ ]:
mu_fixed = 1.0
lambda_seq = np.arange(0.5, 1.0, 0.05)

def mean_wait_at(lam, mu, n_customers=200, B=100):
    return np.mean([simulate_queue(n_customers, lam, mu, rng).mean() for _ in range(B)])

mean_waits = [mean_wait_at(lam, mu_fixed) for lam in lambda_seq]

fig, ax = plt.subplots()
ax.plot(lambda_seq, mean_waits, marker="o")
ax.set_xlabel("lambda")
ax.set_ylabel("mean waiting time")
ax.set_title("Mean wait vs. lambda (mu = 1 fixed)")
plt.show()


Nothing here estimates a parameter -- forward simulation alone already told us something practically important: the system looks fine right up until it doesn't, and the blowup is highly nonlinear. The same kind of payoff shows up in epidemic thresholds, extinction probabilities in population models, and staffing/capacity planning. Sometimes the question really is just: *if the world worked this way, what would happen?*

# Section 2: The SIR model, as another forward simulator

We didn't get to this last time. Three compartments:

$$
S \longrightarrow I \longrightarrow R
$$

Susceptible, Infected/infectious, Recovered/removed. Discrete time steps, population size $N=S_t+I_t+R_t$ fixed, two binomial draws per step:

$$
\Delta I_t \mid S_t, I_t \sim \operatorname{Binomial}\!\Big(S_t, 1-(1-\beta/N)^{I_t}\Big),
\qquad
\Delta R_t \mid I_t \sim \operatorname{Binomial}(I_t, \gamma),
$$
$$
S_{t+1} = S_t - \Delta I_t, \quad I_{t+1} = I_t + \Delta I_t - \Delta R_t, \quad R_{t+1} = R_t + \Delta R_t.
$$

$\beta$ is the transmission parameter, $\gamma$ the recovery parameter. $R_0=\beta/\gamma$ is the basic reproduction number -- roughly, how many new infections one infectious person generates while population immunity is still negligible.

In [ ]:
def simulate_sir(n_steps, beta, gamma, S0, I0, R0, rng):
    N = S0 + I0 + R0
    S = np.empty(n_steps + 1, dtype=int)
    I = np.empty(n_steps + 1, dtype=int)
    R = np.empty(n_steps + 1, dtype=int)
    S[0], I[0], R[0] = S0, I0, R0
    for t in range(n_steps):
        p_infect = 1 - (1 - beta / N) ** I[t]
        delta_I = rng.binomial(S[t], p_infect)
        delta_R = rng.binomial(I[t], gamma)
        S[t + 1] = S[t] - delta_I
        I[t + 1] = I[t] + delta_I - delta_R
        R[t + 1] = R[t] + delta_R
    return S, I, R


## One simulated epidemic

In [ ]:
N = 1000
I0 = 20
gamma_known = 0.15
n_steps = 200

R0_illustrative = 1.5
beta_illustrative = R0_illustrative * gamma_known

S1, I1, R1 = simulate_sir(n_steps, beta_illustrative, gamma_known, N - I0, I0, 0, rng)
t = np.arange(n_steps + 1)

fig, ax = plt.subplots()
ax.plot(t, S1, color="steelblue", label="S")
ax.plot(t, I1, color="firebrick", label="I")
ax.plot(t, R1, color="darkgreen", label="R")
ax.set_xlabel("time")
ax.set_ylabel("count")
ax.set_title("One simulated epidemic (R0 = 1.5)")
ax.legend()
plt.show()


Is this *the* prediction of the model at $R_0=1.5$? Not exactly -- it's one possible realization of what a stochastic model predicts: $Y^{\text{sim}} \sim p(\cdot \mid \theta)$.

## Different parameters, different behavior

Two different $(\beta,\gamma)$ pairs:

In [ ]:
S_low, I_low, R_low = simulate_sir(n_steps, 0.225, gamma_known, N - I0, I0, 0, rng)   # R0 = 1.5
S_high, I_high, R_high = simulate_sir(n_steps, 0.9, gamma_known, N - I0, I0, 0, rng)   # R0 = 6

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(t, I_low, color="firebrick")
axes[0].set_xlabel("time"); axes[0].set_ylabel("I(t)")
axes[0].set_title("beta=0.225, gamma=0.15 (R0=1.5)")
axes[1].plot(t, I_high, color="firebrick")
axes[1].set_xlabel("time"); axes[1].set_ylabel("I(t)")
axes[1].set_title("beta=0.9, gamma=0.15 (R0=6)")
plt.show()


Change the parameters and the whole trajectory changes shape -- size, timing, speed. That's promising for fitting later. But eyeballing whole trajectories doesn't scale to real data. We need a few numbers that track these changes systematically.

## How do we measure "looks like"?

We want to find parameters that make simulated data resemble observed data. How do we measure resemblance? For today, a first idea: hand-pick summary statistics that seem to vary with the parameters.

Two candidates for today: **final epidemic size** (how many people were ever infected) and **early growth rate** (slope of $\log I(t)$ over the first few steps). Generically, $s(Y)$.

In [ ]:
window = 8  # early-growth-rate fitting window

def sir_summary_once(R0, rng):
    beta = R0 * gamma_known
    S, I, R = simulate_sir(n_steps, beta, gamma_known, N - I0, I0, 0, rng)
    cum_infected = N - S
    final_size = cum_infected[-1]
    I_early = I[: window + 1].astype(float)
    t_early = np.arange(window + 1)
    growth_rate = np.polyfit(t_early, np.log(np.maximum(I_early, 1)), 1)[0]
    return np.array([final_size, growth_rate])


If the simulated dataset is random, its summary is random too.

## From two parameters to one

For the systematic check that follows, fix $\gamma$ (say, known from clinical data on infectious duration) and vary only $R_0=\beta/\gamma$. One unknown parameter instead of two -- simpler to explore now, and simpler to fit later.

Does a candidate summary actually help distinguish different values of $R_0$? Look at its distribution across a range of $R_0$'s.

In [ ]:
R0_display = [2, 3, 4, 5, 6, 7, 8]
growth_by_r0 = [[sir_summary_once(r, rng)[1] for _ in range(150)] for r in R0_display]
final_by_r0 = [[sir_summary_once(r, rng)[0] for _ in range(150)] for r in R0_display]


## A useful summary: early growth rate

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(growth_by_r0, labels=R0_display)
ax.set_xlabel("R0")
ax.set_ylabel("early growth rate")
ax.set_title("Growth rate: centers move, little overlap")
plt.show()


## A weaker summary: final epidemic size

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(final_by_r0, labels=R0_display)
ax.set_xlabel("R0")
ax.set_ylabel("final size")
ax.set_title("Final size: centers move, then flatten and overlap")
plt.show()


If you observed one epidemic, which statistic would tell you more about $R_0$? Growth rate keeps separating across the whole range; final size stops moving once most of the population is going to be infected either way.

# Section 3: Turn the simulator around

We now have a good sense of how this simulator behaves -- which summaries move, which don't. Time to use that to fit parameters to real data.

Forward: $R_0 \to$ simulate epidemic $\to s(Y)$. Now reverse it: $s(Y_{\text{obs}}) \to$ what $R_0$ could have produced this?

The algorithm you'd invent yourself:

1. choose a candidate $R_0$;
2. simulate many epidemics;
3. compute the same summary for each;
4. average the simulated summaries;
5. compare with the observed summary;
6. try another $R_0$.

Which candidate makes the simulated summaries look most like the observed one?

# Section 4: Formalize SMM

$$
s_{\text{obs}} = s(Y_{\text{obs}}), \qquad
\hat m_B(\theta) = \frac1B\sum_{b=1}^B s(Y_\theta^{(b)}) \approx E_\theta[s(Y)].
$$

For one summary:
$$
Q(\theta) = \big[s_{\text{obs}} - \hat m_B(\theta)\big]^2,
\qquad
\hat\theta = \arg\min_\theta Q(\theta).
$$

## Objective-function plot: growth rate alone

In [ ]:
R0_true = 6.0
R0_grid = np.arange(4.0, 8.01, 0.1)
model_grid = np.array([
    np.mean([sir_summary_once(r, rng) for _ in range(300)], axis=0) for r in R0_grid
])

obs_illustrative = sir_summary_once(R0_true, rng)

Q_growth_only = (model_grid[:, 1] - obs_illustrative[1]) ** 2

fig, ax = plt.subplots()
ax.plot(R0_grid, Q_growth_only)
ax.axvline(R0_true, color="blue", linestyle="--", label="true R0")
ax.axvline(R0_grid[np.argmin(Q_growth_only)], color="red", label="estimated R0")
ax.set_xlabel("R0")
ax.set_ylabel("Q(R0)")
ax.set_title("Objective function, growth rate alone")
ax.legend()
plt.show()


**Simulated Method of Moments**: adjust parameters until simulated summaries match observed summaries. "Moment" here just means a chosen numerical summary of the data, not the classical statistical method-of-moments machinery.

# Section 5: What if we use more than one summary?

$$
s(Y) = \begin{pmatrix}s_1(Y) \\ s_2(Y)\end{pmatrix},
\qquad
g(\theta) = s_{\text{obs}} - \hat m_B(\theta).
$$

How should two discrepancies become one number?

## First attempt: add squared discrepancies

$$
Q(\theta) = g_1(\theta)^2 + g_2(\theta)^2.
$$

This looks natural. It is not generally a good idea.

To do better we need an estimate of $\operatorname{Cov}(g(\theta))$. As a shortcut today, we estimate it using simulations at the true $R_0$ (in practice you don't know $R_0$ in advance: fit once with rough weighting, estimate this near that estimate, then refit).

In [ ]:
B_omega = 500
at_true = np.array([sir_summary_once(R0_true, rng) for _ in range(B_omega)])
Omega = np.cov(at_true.T)


## Four ways to combine (or not combine) the two summaries

In [ ]:
Omega_inv = np.linalg.inv(Omega)

def fit_once(seed):
    local_rng = np.random.default_rng(seed)
    obs = sir_summary_once(R0_true, local_rng)  # one fresh "observed" epidemic
    g = model_grid - obs

    Q_strong = g[:, 1] ** 2
    Q_weak = g[:, 0] ** 2
    Q_unweighted = (g ** 2).sum(axis=1)
    Q_weighted = np.einsum("ij,jk,ik->i", g, Omega_inv, g)

    return np.array([
        R0_grid[np.argmin(Q_strong)],
        R0_grid[np.argmin(Q_weak)],
        R0_grid[np.argmin(Q_unweighted)],
        R0_grid[np.argmin(Q_weighted)],
    ])


## Discrepancies should be judged relative to ordinary variability

A discrepancy of 10 is huge if the statistic normally varies by 1, and trivial if it normally varies by 100. For independent summaries:

$$
Q(\theta) = \frac{g_1(\theta)^2}{\sigma_1^2} + \frac{g_2(\theta)^2}{\sigma_2^2}.
$$

More generally, with possibly correlated summaries, use the full covariance matrix:

$$
Q_W(\theta) = g(\theta)^\top W g(\theta), \qquad W \approx \operatorname{Cov}(g)^{-1}.
$$

In [ ]:
print(Omega)  # diagonal: variability of each summary; off-diagonal: correlation


Diagonal terms handle differences in variability; off-diagonal terms account for correlation between summaries -- final size and growth rate are not independent.

Weighting does **not** make every summary equally informative. It puts discrepancies on an appropriate stochastic scale; how much a summary then contributes still depends on how strongly its mean responds to the parameter (the derivative) -- a property weighting can't manufacture.

## Does it actually help? (100 repeated fits, one fresh epidemic each)

In [ ]:
fits = np.array([fit_once(seed) for seed in range(100)])
labels = ["strong", "weak", "unweighted", "weighted"]

bias = fits.mean(axis=0) - R0_true
rmse = np.sqrt(((fits - R0_true) ** 2).mean(axis=0))
for lab, b, r in zip(labels, bias, rmse):
    print(f"{lab:12s} bias={b:6.3f}  rmse={r:6.3f}")


Read this in order:

- **Weak alone** (final size) does clearly worse than **strong alone** (growth rate) -- matches the information calculation.
- **Unweighted** performs *identically* to weak alone: final size's raw scale (hundreds of people) so overwhelms growth rate's (fractions of a unit) that growth rate might as well not be in the objective at all.
- **Weighted** ($\hat\Omega^{-1}$) recovers the lost information -- it matches or beats even the best single summary.

A weak/noisy summary can hurt if combined carelessly; weighting keeps it from having inappropriate influence.

# Section 6: The limits of weighting

But weighting is not enough on its own. What if a summary contains almost no information about the parameter? What if two summaries contain essentially the same information? What if our summaries miss an important feature of the data entirely? No weighting matrix can rescue a bad set of summaries.

$$
\boxed{
\theta \to \text{simulate many datasets} \to \text{calculate summaries} \to \text{estimate expected summaries}
}
$$
$$
\boxed{
\text{choose } \theta \text{ so simulated summaries resemble observed summaries}
}
$$

SMM turns a forward simulator into a parameter-estimation method.